<a href="https://colab.research.google.com/github/kader-xai/llm-from-scratch/blob/main/notebooks/todo/Chapter13_generate_todo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🧱 **LLM From Scratch · Chapter 13 — Generate — sample text from your model**

> ✍️ TODO notebook — fill in the `# YOUR CODE HERE` blanks, then check against the answer.

## Goal

The payoff. Feed the trained model a starting token and let it autoregressively sample new
characters one at a time. Right after training (Chapter 12) run this in the same session so `model`,
`decode`, and `device` are in scope. We also explore **temperature** — the knob that trades coherence
for creativity.

> ℹ️ This chapter assumes you've just run **Chapter 12** and have a trained `model`,
> plus `decode` and `device` in memory. If not, train the model first.

In [ ]:
# Greedy-ish sampling from the trained model
start = torch.zeros((1, 1), dtype=torch.long, device=device)
# YOUR CODE HERE: call model.generate(...) for 400 new tokens, then decode the result
generated = ...
print(decode(generated[0].tolist()))

## Temperature — coherence vs creativity

Divide logits by `T` before softmax. `T<1` sharpens (safer, repetitive); `T>1` flattens (wilder).
This is the single most useful generation knob.

In [ ]:
@torch.no_grad()
def generate_temp(model, start, n, temperature=1.0):
    ids = start
    for _ in range(n):
        logits, _ = model(ids[:, -model.block_size:])
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        ids = torch.cat([ids, torch.multinomial(probs, 1)], dim=1)
    return ids

for T in (0.5, 1.0, 1.3):
    out = generate_temp(model, torch.zeros((1,1), dtype=torch.long, device=device), 200, T)
    print(f"\n=== temperature {T} ===")
    print(decode(out[0].tolist()))

## 🎉 You built and trained an LLM from scratch

You now have, in pure PyTorch and your own hands:
- a tokenizer + dataloader (Ch 01)
- token + position embeddings (Ch 02-04)
- LayerNorm, attention, multi-head attention, feed-forward (Ch 05-08)
- the transformer block and the full GPT (Ch 09-11)
- a training loop and a text sampler (Ch 12-13)

**Where to go next** (see the README roadmap): swap the char tokenizer for **tiktoken/BPE**, scale
the model (more layers/heads/`C`), train longer on a bigger corpus (TinyStories), add **RoPE**
positional encoding, then **instruction-tune** it into a chat model.

For the *production* versions of every idea here — fine-tuning, RAG, alignment, edge deployment,
mechanistic interpretability — see the companion **[Data Science Roadmap](https://github.com/kader-xai/data-science-roadmap)** (Parts 8, 12, 15).